In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell F4.1 - Overview, paths, and fixed manuscript design
# Purpose:
# Create the two manuscript panels for Figure 4 from the completed Notebook 25 results.
#
# Figure 4A:
# Final combined heatmap of 14 supported mapped subwindows,
# 5 supported non-coordinate groups, and P7319/P2846 carrier states.
#
# Figure 4B:
# Quantitative summary of the 14 mapped subwindows and 5 non-coordinate groups.
# Mapped subwindows are positioned by MG1655 chromosomal location.
# Non-coordinate groups are displayed separately as categorical groups.
# Point = median matched-comparator Jaccard dissimilarity across the 16 high-MIC pathogens.
# Vertical bar = interquartile range.
# Marker size = ablation drop from Notebook 21 (mapped) or Notebook 22 (non-coordinate).
# P7319/P2846 are not included because they are binary carrier states rather than
# ablation-defined sequence groups with directly comparable quantitative summaries.
#
# Manuscript figure rule:
# - minimal text inside the panels;
# - grayscale-compatible;
# - no unnecessary legend;
# - interpretation belongs in the figure caption.

from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
PROJECT_ROOT = _repo_root()

RESULTS_TABLE_DIR = (
    PROJECT_ROOT
    / "05_Results"
    / "Tables"
)

NB25_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "25_High_MIC_Matched_Comparator_Locus_Distribution"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "06_Manuscript"
    / "Figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HIGH_MATRIX_FILE = (
    NB25_DIR
    / "25_high_MIC_matched_comparator_dissimilarity_matrices.npz"
)

FINAL_SUMMARY_FILE = (
    RESULTS_TABLE_DIR
    / "25_final_locus_distribution_summary.csv"
)

FIG4A_PNG = (
    FIGURE_DIR
    / "Figure4A_high_MIC_matched_comparator_locus_dissimilarity.png"
)

FIG4A_PDF = (
    FIGURE_DIR
    / "Figure4A_high_MIC_matched_comparator_locus_dissimilarity.pdf"
)

FIG4B_PNG = (
    FIGURE_DIR
    / "Figure4B_mapped_noncoordinate_sequence_summary.png"
)

FIG4B_PDF = (
    FIGURE_DIR
    / "Figure4B_mapped_noncoordinate_sequence_summary.pdf"
)

for path in [
    HIGH_MATRIX_FILE,
    FINAL_SUMMARY_FILE,
]:
    assert path.exists(), (
        f"Required input not found: {path}"
    )

print("Figure 4 manuscript notebook")
print("Figure output folder:", FIGURE_DIR)

print(
    "\nTransition: Cell F4.2 will load and verify the frozen Notebook 25 results."
)


In [ ]:
#@title Cell F4.2 - Load and verify frozen Notebook 25 results
# Purpose:
# Use only the final saved Notebook 25 outputs.
# No analysis is recomputed here.

with np.load(
    HIGH_MATRIX_FILE
) as archive:
    high_mean_dissimilarity = np.asarray(
        archive[
            "high_mean_dissimilarity"
        ],
        dtype=float,
    )

    high_min_dissimilarity = np.asarray(
        archive[
            "high_min_dissimilarity"
        ],
        dtype=float,
    )

    high_max_dissimilarity = np.asarray(
        archive[
            "high_max_dissimilarity"
        ],
        dtype=float,
    )

    high_sample_index = np.asarray(
        archive[
            "high_sample_index"
        ],
        dtype=int,
    )

    high_log2_mic = np.asarray(
        archive[
            "high_log2_mic"
        ],
        dtype=float,
    )

    region_names = np.asarray(
        archive[
            "region_names"
        ]
    ).astype(
        str
    )

final_summary = pd.read_csv(
    FINAL_SUMMARY_FILE
)

assert high_mean_dissimilarity.shape == (
    16,
    14,
)

assert high_min_dissimilarity.shape == (
    16,
    14,
)

assert high_max_dissimilarity.shape == (
    16,
    14,
)

assert len(
    high_sample_index
) == 16

assert len(
    high_log2_mic
) == 16

assert len(
    region_names
) == 14

assert len(
    final_summary
) == 14

assert np.array_equal(
    region_names,
    final_summary[
        "subwindow_name"
    ].astype(
        str
    ).to_numpy(),
), (
    "Region order differs between the frozen matrix and final summary."
)

required_summary_columns = [
    "subwindow_name",
    "reference_midpoint_Mb",
    "Q1_dissimilarity",
    "median_dissimilarity",
    "Q3_dissimilarity",
    "Notebook21_observed_drop_after_removal",
]

for column in required_summary_columns:
    assert column in final_summary.columns, (
        f"Missing required final-summary column: {column}"
    )

top_median = (
    final_summary
    .sort_values(
        "median_dissimilarity",
        ascending=False,
    )
    .iloc[0]
)

top_ablation = (
    final_summary
    .sort_values(
        "Notebook21_observed_drop_after_removal",
        ascending=False,
    )
    .iloc[0]
)

assert (
    top_median[
        "subwindow_name"
    ]
    == "W10_S08"
)

assert (
    top_ablation[
        "subwindow_name"
    ]
    == "W10_S08"
)

print("Frozen Notebook 25 inputs: PASS")
print("High-MIC pathogens:", high_mean_dissimilarity.shape[0])
print("Supported mapped loci:", high_mean_dissimilarity.shape[1])
print(
    "Top locus by median dissimilarity:",
    top_median["subwindow_name"],
)
print(
    "Top locus by ablation drop:",
    top_ablation["subwindow_name"],
)

print(
    "\nTransition: Cell F4.3 will create manuscript Figure 4A."
)


In [ ]:
#@title Cell F4.3 - Figure 4A high-MIC locus-distribution heatmap
# Purpose:
# Show the regional sequence difference between each high-MIC pathogen
# and its 3 matched comparators across the 14 supported mapped loci.
#
# Rows are already ordered from highest to lower MIC in the frozen Notebook 25 output.
# Minimal manuscript presentation:
# - no title;
# - row labels H01-H16 only;
# - short locus labels;
# - grayscale;
# - short colorbar label.

row_labels = [
    f"H{i:02d}"
    for i in range(
        1,
        17,
    )
]

fig, ax = plt.subplots(
    figsize=(10.5, 6.7)
)

image = ax.imshow(
    high_mean_dissimilarity,
    aspect="auto",
    interpolation="nearest",
    cmap="Greys",
    vmin=0.0,
    vmax=float(
        np.max(
            high_mean_dissimilarity
        )
    ),
)

ax.set_xlabel(
    "Chromosomal subwindow"
)

ax.set_ylabel(
    "High-MIC pathogen"
)

ax.set_xticks(
    np.arange(
        len(
            region_names
        )
    )
)

ax.set_xticklabels(
    region_names,
    rotation=90,
)

ax.set_yticks(
    np.arange(
        len(
            row_labels
        )
    )
)

ax.set_yticklabels(
    row_labels
)

parent_names = np.array(
    [
        name.split(
            "_"
        )[0]
        for name in region_names
    ]
)

for i in range(
    1,
    len(
        parent_names
    ),
):
    if (
        parent_names[i]
        != parent_names[i - 1]
    ):
        ax.axvline(
            i - 0.5,
            linewidth=0.8,
        )

cbar = fig.colorbar(
    image,
    ax=ax,
)

cbar.set_label(
    "Mean Jaccard dissimilarity"
)

fig.tight_layout()

fig.savefig(
    FIG4A_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG4A_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print("Figure 4A created.")
print(FIG4A_PNG)

print(
    "\nTransition: Cell F4.4 will prepare the mapped Figure 4B summary inputs."
)


In [ ]:
#@title Cell F4.4 - Prepare mapped Figure 4B summary inputs
# Purpose:
# Prepare the quantitative summaries for the 14 supported mapped subwindows.
# The final Figure 4B is drawn only after the five non-coordinate groups have
# been verified and their matched-comparator dissimilarities have been calculated.
#
# No new analysis is performed here.

mapped_x = final_summary[
    "reference_midpoint_Mb"
].to_numpy(
    dtype=float
)

mapped_median = final_summary[
    "median_dissimilarity"
].to_numpy(
    dtype=float
)

mapped_q1 = final_summary[
    "Q1_dissimilarity"
].to_numpy(
    dtype=float
)

mapped_q3 = final_summary[
    "Q3_dissimilarity"
].to_numpy(
    dtype=float
)

mapped_ablation_drop = final_summary[
    "Notebook21_observed_drop_after_removal"
].to_numpy(
    dtype=float
)

mapped_lower_error = (
    mapped_median
    - mapped_q1
)

mapped_upper_error = (
    mapped_q3
    - mapped_median
)

assert len(mapped_x) == 14
assert len(mapped_median) == 14
assert len(mapped_q1) == 14
assert len(mapped_q3) == 14
assert len(mapped_ablation_drop) == 14

assert np.all(
    mapped_lower_error >= -1e-12
)

assert np.all(
    mapped_upper_error >= -1e-12
)

print(
    "Mapped Figure 4B inputs: PASS"
)

print(
    "Mapped subwindows:",
    len(mapped_x),
)

print(
    "\nTransition: Cell F4.5 will verify the five supported non-coordinate groups."
)


In [ ]:
#@title Cell F4.5 - Verify the five supported non-coordinate groups
# Purpose:
# Inspect the exact Notebook 22 outputs before expanding Figure 4A.
#
# No new analysis is performed here.

from pathlib import Path
import numpy as np
import pandas as pd

NB22_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "22_Unmapped_Multimapped_Carrier_Pattern_Refinement"
)

NB22_ASSIGNMENT = (
    NB22_DIR
    / "22_carrier_pattern_cluster_assignment.npz"
)

NB22_RESULTS = (
    RESULTS_TABLE_DIR
    / "22_carrier_pattern_cluster_matched_random_results.csv"
)

SUPPORTED_NONCOORDINATE = [
    "UNMAPPED_C02",
    "UNMAPPED_C03",
    "UNMAPPED_C05",
    "MULTIMAPPED_C05",
    "MULTIMAPPED_C07",
]

assert NB22_ASSIGNMENT.exists(), (
    f"Missing Notebook 22 assignment file: {NB22_ASSIGNMENT}"
)

assert NB22_RESULTS.exists(), (
    f"Missing Notebook 22 result file: {NB22_RESULTS}"
)

# ------------------------------------------------------------
# Inspect assignment file
# ------------------------------------------------------------

print("Notebook 22 assignment NPZ keys:")

with np.load(
    NB22_ASSIGNMENT,
    allow_pickle=True,
) as archive:

    for key in archive.files:

        arr = np.asarray(
            archive[key]
        )

        print(
            f"  {key}: shape={arr.shape}, dtype={arr.dtype}"
        )

# ------------------------------------------------------------
# Verify supported cluster names
# ------------------------------------------------------------

nb22_results = pd.read_csv(
    NB22_RESULTS
)

print(
    "\nNotebook 22 result columns:"
)

print(
    list(
        nb22_results.columns
    )
)

# Find the column that actually contains the cluster names.
matching_name_columns = []

for column in nb22_results.columns:

    values = set(
        nb22_results[
            column
        ]
        .astype(str)
        .tolist()
    )

    if set(
        SUPPORTED_NONCOORDINATE
    ).issubset(
        values
    ):
        matching_name_columns.append(
            column
        )

assert len(
    matching_name_columns
) == 1, (
    "Could not uniquely identify the Notebook 22 cluster-name column.\n"
    f"Matching columns: {matching_name_columns}"
)

CLUSTER_NAME_COLUMN = (
    matching_name_columns[0]
)

supported_rows = (
    nb22_results.loc[
        nb22_results[
            CLUSTER_NAME_COLUMN
        ].astype(str).isin(
            SUPPORTED_NONCOORDINATE
        )
    ]
    .copy()
)

assert len(
    supported_rows
) == 5

print(
    "\nCluster-name column:",
    CLUSTER_NAME_COLUMN
)

print(
    "\nFive supported non-coordinate groups:"
)

display(
    supported_rows
)

print(
    "\nCell F4.5 complete."
)

print(
    "Next step: use the verified Notebook 22 assignment fields "
    "to calculate H01-H16 matched-comparator dissimilarity "
    "for these five groups."
)

In [ ]:
#@title Cell F4.6 - Matched-comparator dissimilarity for five non-coordinate groups
# Purpose:
# Calculate the same mean Jaccard dissimilarity used in Figure 4A,
# now for the five supported non-coordinate groups from Notebook 22.
#
# The authoritative matching file is:
# 03_matched_pairs_16x3.csv
#
# No new association test is performed.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse

# ------------------------------------------------------------
# Exact inputs
# ------------------------------------------------------------

UNITIG_MATRIX = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "10_Whole_Chromosome_Unitigs"
    / "10_variable_unitig_matrix_176xM.npz"
)

UNITIG_SAMPLES = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "10_Whole_Chromosome_Unitigs"
    / "10_unitig_sample_order.csv"
)

NB22_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "22_Unmapped_Multimapped_Carrier_Pattern_Refinement"
)

NB22_ASSIGNMENT = (
    NB22_DIR
    / "22_carrier_pattern_cluster_assignment.npz"
)

NB22_RESULTS = (
    RESULTS_TABLE_DIR
    / "22_carrier_pattern_cluster_matched_random_results.csv"
)

for path in [
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
    NB22_ASSIGNMENT,
    NB22_RESULTS,
]:
    assert path.exists(), (
        f"Required input not found: {path}"
    )

# ------------------------------------------------------------
# Locate the one authoritative matching file
# ------------------------------------------------------------

matching_candidates = list(
    PROJECT_ROOT.rglob(
        "03_matched_pairs_16x3.csv"
    )
)

assert len(
    matching_candidates
) == 1, (
    "Expected exactly one authoritative "
    "03_matched_pairs_16x3.csv.\n"
    f"Found: {matching_candidates}"
)

MATCHED_PAIRS_FILE = (
    matching_candidates[0]
)

print(
    "Authoritative matching file:"
)

print(
    MATCHED_PAIRS_FILE
)

# ------------------------------------------------------------
# Load 176-pathogen sample order
# ------------------------------------------------------------

samples = (
    pd.read_csv(
        UNITIG_SAMPLES
    )
    .sort_values(
        "sample_index"
    )
    .reset_index(
        drop=True
    )
)

assert len(
    samples
) == 176

assert "biosample" in samples.columns
assert "sample_index" in samples.columns

biosample_to_index = dict(
    zip(
        samples[
            "biosample"
        ]
        .astype(str)
        .str.strip(),
        samples[
            "sample_index"
        ].astype(int),
    )
)

# ------------------------------------------------------------
# Load and verify fixed 48 matched pairs
# ------------------------------------------------------------

matched_pairs_raw = pd.read_csv(
    MATCHED_PAIRS_FILE
)

required_columns = [
    "upper_biosample",
    "comparison_biosample",
]

for column in required_columns:
    assert column in matched_pairs_raw.columns

assert len(
    matched_pairs_raw
) == 48

matched_pairs_48 = matched_pairs_raw[
    [
        "upper_biosample",
        "comparison_biosample",
    ]
].copy()

matched_pairs_48[
    "high_sample_index"
] = (
    matched_pairs_48[
        "upper_biosample"
    ]
    .astype(str)
    .str.strip()
    .map(
        biosample_to_index
    )
)

matched_pairs_48[
    "comparator_sample_index"
] = (
    matched_pairs_48[
        "comparison_biosample"
    ]
    .astype(str)
    .str.strip()
    .map(
        biosample_to_index
    )
)

assert matched_pairs_48[
    "high_sample_index"
].notna().all()

assert matched_pairs_48[
    "comparator_sample_index"
].notna().all()

matched_pairs_48[
    "high_sample_index"
] = matched_pairs_48[
    "high_sample_index"
].astype(int)

matched_pairs_48[
    "comparator_sample_index"
] = matched_pairs_48[
    "comparator_sample_index"
].astype(int)

assert (
    matched_pairs_48[
        "high_sample_index"
    ].nunique()
    == 16
)

assert (
    matched_pairs_48[
        "comparator_sample_index"
    ].nunique()
    == 23
)

pairs_per_high = (
    matched_pairs_48
    .groupby(
        "high_sample_index"
    )
    .size()
)

assert (
    pairs_per_high
    == 3
).all()

# Critical identity check:
# must be exactly the same H01-H16 pathogens used in Figure 4A.
assert set(
    matched_pairs_48[
        "high_sample_index"
    ].tolist()
) == set(
    high_sample_index.astype(
        int
    ).tolist()
)

print(
    "\nMatched-pair QC: PASS"
)

print(
    "Pairs:",
    len(
        matched_pairs_48
    )
)

print(
    "High-MIC pathogens:",
    matched_pairs_48[
        "high_sample_index"
    ].nunique()
)

print(
    "Unique matched comparators:",
    matched_pairs_48[
        "comparator_sample_index"
    ].nunique()
)

# ------------------------------------------------------------
# Load unitig matrix
# ------------------------------------------------------------

X = sparse.load_npz(
    UNITIG_MATRIX
).tocsc()

assert X.shape == (
    176,
    1_287_844,
)

# ------------------------------------------------------------
# Load Notebook 22 cluster assignment
# ------------------------------------------------------------

with np.load(
    NB22_ASSIGNMENT
) as archive:

    cluster_assignment = np.asarray(
        archive[
            "cluster_assignment"
        ],
        dtype=np.int16,
    )

assert cluster_assignment.shape == (
    1_287_844,
)

nb22_results = pd.read_csv(
    NB22_RESULTS
)

SUPPORTED_NONCOORDINATE = [
    "UNMAPPED_C02",
    "UNMAPPED_C03",
    "UNMAPPED_C05",
    "MULTIMAPPED_C05",
    "MULTIMAPPED_C07",
]

supported_noncoordinate = (
    nb22_results.loc[
        nb22_results[
            "cluster_name"
        ].astype(str).isin(
            SUPPORTED_NONCOORDINATE
        ),
        [
            "global_cluster_code",
            "parent_group_name",
            "cluster_name",
            "n_unitigs",
            "observed_drop_after_removal",
        ],
    ]
    .copy()
)

assert len(
    supported_noncoordinate
) == 5

# Fixed display order:
# UNMAPPED first, then MULTIMAPPED.
preferred_order = [
    "UNMAPPED_C02",
    "UNMAPPED_C03",
    "UNMAPPED_C05",
    "MULTIMAPPED_C05",
    "MULTIMAPPED_C07",
]

order_map = {
    name: i
    for i, name in enumerate(
        preferred_order
    )
}

supported_noncoordinate[
    "plot_order"
] = supported_noncoordinate[
    "cluster_name"
].map(
    order_map
)

assert supported_noncoordinate[
    "plot_order"
].notna().all()

supported_noncoordinate = (
    supported_noncoordinate
    .sort_values(
        "plot_order"
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------------------------
# Calculate 16 x 5 matched-comparator dissimilarity matrix
# ------------------------------------------------------------

noncoordinate_mean_dissimilarity = np.zeros(
    (
        16,
        5,
    ),
    dtype=float,
)

for cluster_index, cluster_row in supported_noncoordinate.iterrows():

    global_code = int(
        cluster_row[
            "global_cluster_code"
        ]
    )

    cluster_name = str(
        cluster_row[
            "cluster_name"
        ]
    )

    unitig_indices = np.flatnonzero(
        cluster_assignment
        == global_code
    )

    assert len(
        unitig_indices
    ) == int(
        cluster_row[
            "n_unitigs"
        ]
    ), (
        f"{cluster_name}: expected "
        f"{int(cluster_row['n_unitigs'])} unitigs, "
        f"found {len(unitig_indices)}."
    )

    for high_row, high_index in enumerate(
        high_sample_index.astype(
            int
        )
    ):

        comparator_indices = (
            matched_pairs_48.loc[
                matched_pairs_48[
                    "high_sample_index"
                ]
                == high_index,
                "comparator_sample_index",
            ]
            .to_numpy(
                dtype=int
            )
        )

        assert len(
            comparator_indices
        ) == 3

        high_vector = X[
            high_index,
            unitig_indices,
        ]

        high_present = float(
            high_vector.sum()
        )

        dissimilarities = []

        for comparator_index in comparator_indices:

            comparator_vector = X[
                comparator_index,
                unitig_indices,
            ]

            comparator_present = float(
                comparator_vector.sum()
            )

            shared_present = float(
                high_vector.multiply(
                    comparator_vector
                ).sum()
            )

            union_present = (
                high_present
                + comparator_present
                - shared_present
            )

            if union_present > 0:
                similarity = (
                    shared_present
                    / union_present
                )
            else:
                similarity = 1.0

            dissimilarities.append(
                1.0
                - similarity
            )

        noncoordinate_mean_dissimilarity[
            high_row,
            cluster_index,
        ] = float(
            np.mean(
                dissimilarities
            )
        )

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

noncoordinate_table = pd.DataFrame(
    noncoordinate_mean_dissimilarity,
    index=[
        f"H{i:02d}"
        for i in range(
            1,
            17,
        )
    ],
    columns=preferred_order,
)

print(
    "\nNon-coordinate matched-comparator calculation: PASS"
)

print(
    "\nAll 16 high-MIC pathogens:"
)

display(
    noncoordinate_table
)

print(
    "\nH01, H02 and H15:"
)

display(
    noncoordinate_table.loc[
        [
            "H01",
            "H02",
            "H15",
        ]
    ]
)

# ------------------------------------------------------------
# Save for expanded Figure 4A
# ------------------------------------------------------------

NONCOORDINATE_MATRIX_FILE = (
    NB25_DIR
    / "Figure4_noncoordinate_high_MIC_matched_dissimilarity.npz"
)

np.savez_compressed(
    NONCOORDINATE_MATRIX_FILE,
    noncoordinate_mean_dissimilarity=(
        noncoordinate_mean_dissimilarity
    ),
    high_sample_index=high_sample_index.astype(
        np.int16
    ),
    high_log2_mic=high_log2_mic,
    cluster_names=np.asarray(
        preferred_order,
        dtype="U24",
    ),
)

print(
    "\nSaved:"
)

print(
    NONCOORDINATE_MATRIX_FILE
)

print(
    "\nCell F4.6 complete."
)

print(
    "Next step: inspect H01, H02 and H15 before modifying Figure 4A."
)

In [ ]:
#@title Cell F4.7 - Check rare higher-MIC patterns in H01, H02 and H15
# Purpose:
# Determine whether H01, H02 or H15 carry BH-significant higher-MIC
# patterns 7319 and/or 2846.
#
# No new association analysis is performed.

from pathlib import Path
import numpy as np
import pandas as pd

NB11_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "11_Unitig_Patterns"
)

PATTERN_WORDS_FILE = (
    NB11_DIR
    / "11_unique_pattern_words.npz"
)

PATTERN_SAMPLE_ORDER = (
    NB11_DIR
    / "11_pattern_sample_order.csv"
)

TARGET_PATTERNS = [
    7319,
    2846,
]

for path in [
    PATTERN_WORDS_FILE,
    PATTERN_SAMPLE_ORDER,
]:
    assert path.exists(), (
        f"Required input not found: {path}"
    )

sample_order = (
    pd.read_csv(
        PATTERN_SAMPLE_ORDER
    )
    .sort_values(
        "sample_index"
    )
    .reset_index(
        drop=True
    )
)

assert len(
    sample_order
) == 176

assert np.array_equal(
    sample_order[
        "sample_index"
    ].to_numpy(
        dtype=int
    ),
    np.arange(
        176
    ),
)

# Verify sample order matches the Figure 4 sample order.
assert set(
    high_sample_index.astype(
        int
    ).tolist()
).issubset(
    set(
        sample_order[
            "sample_index"
        ].astype(
            int
        ).tolist()
    )
)

# ------------------------------------------------------------
# Load exact 176-bit carrier patterns
# ------------------------------------------------------------

with np.load(
    PATTERN_WORDS_FILE
) as archive:
    print(
        "Pattern-word NPZ keys:"
    )

    for key in archive.files:
        arr = np.asarray(
            archive[
                key
            ]
        )

        print(
            f"  {key}: shape={arr.shape}, dtype={arr.dtype}"
        )

    candidate_arrays = []

    for key in archive.files:
        arr = np.asarray(
            archive[
                key
            ]
        )

        if (
            arr.ndim == 2
            and arr.shape[0] == 504_889
            and arr.shape[1] == 3
            and np.issubdtype(
                arr.dtype,
                np.unsignedinteger,
            )
        ):
            candidate_arrays.append(
                (
                    key,
                    arr,
                )
            )

assert len(
    candidate_arrays
) == 1, (
    "Could not uniquely identify the 504,889 x 3 uint64 pattern-word array.\n"
    f"Candidates: {[x[0] for x in candidate_arrays]}"
)

PATTERN_WORD_KEY, pattern_words = candidate_arrays[0]

print(
    "\nAuthoritative pattern-word key:",
    PATTERN_WORD_KEY
)

# ------------------------------------------------------------
# Decode carriers for the two rare significant patterns
# ------------------------------------------------------------

def decode_pattern_words(
    words,
    n_samples=176,
):
    carrier = np.zeros(
        n_samples,
        dtype=np.uint8,
    )

    for sample_index in range(
        n_samples
    ):
        word_index = (
            sample_index
            // 64
        )

        bit_index = (
            sample_index
            % 64
        )

        carrier[
            sample_index
        ] = (
            int(
                words[
                    word_index
                ]
            )
            >> bit_index
        ) & 1

    return carrier


carrier_rows = []

for pattern_id in TARGET_PATTERNS:

    words = pattern_words[
        pattern_id,
        :
    ]

    carrier = decode_pattern_words(
        words
    )

    carrier_indices = np.flatnonzero(
        carrier == 1
    )

    carrier_rows.append(
        {
            "pattern_id": pattern_id,
            "carrier_count": len(
                carrier_indices
            ),
            "carrier_sample_indices": carrier_indices.tolist(),
        }
    )

    print(
        f"\nPattern {pattern_id}"
    )

    print(
        "Carrier count:",
        len(
            carrier_indices
        )
    )

    print(
        "Carrier sample indices:",
        carrier_indices.tolist()
    )

# ------------------------------------------------------------
# H01, H02 and H15
# ------------------------------------------------------------

selected_labels = [
    "H01",
    "H02",
    "H15",
]

selected_rows = [
    0,
    1,
    14,
]

selected_high_indices = {
    label: int(
        high_sample_index[
            row
        ]
    )
    for label, row in zip(
        selected_labels,
        selected_rows,
    )
}

check_rows = []

for pattern_id in TARGET_PATTERNS:

    carrier = decode_pattern_words(
        pattern_words[
            pattern_id,
            :
        ]
    )

    for label, sample_index in selected_high_indices.items():

        check_rows.append(
            {
                "high_label": label,
                "sample_index": sample_index,
                "pattern_id": pattern_id,
                "carrier": bool(
                    carrier[
                        sample_index
                    ]
                ),
            }
        )

check_table = pd.DataFrame(
    check_rows
)

print(
    "\nH01, H02 and H15 carrier status:"
)

display(
    check_table
)

print(
    "\nCell F4.7 complete."
)

In [ ]:
#@title Cell F4.8 - Verify the existing Figure 4A mapped matrix
# Purpose:
# Identify the exact 16 x 14 mapped-subwindow matrix already used
# for Figure 4A before combining it with the new non-coordinate results.
#
# No analysis is performed.

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Find existing 16 x 14 numeric arrays in the notebook
# ------------------------------------------------------------

candidate_arrays = []

for name, obj in list(globals().items()):

    if isinstance(obj, np.ndarray):

        if (
            obj.ndim == 2
            and obj.shape == (16, 14)
            and np.issubdtype(
                obj.dtype,
                np.number,
            )
        ):
            candidate_arrays.append(
                (
                    name,
                    obj.dtype,
                    float(
                        np.nanmin(obj)
                    ),
                    float(
                        np.nanmax(obj)
                    ),
                )
            )

print(
    "Existing 16 x 14 numeric arrays:"
)

candidate_table = pd.DataFrame(
    candidate_arrays,
    columns=[
        "variable_name",
        "dtype",
        "minimum",
        "maximum",
    ],
)

display(
    candidate_table
)

# ------------------------------------------------------------
# Find existing 14-element label objects
# ------------------------------------------------------------

candidate_labels = []

for name, obj in list(globals().items()):

    try:
        values = list(
            obj
        )
    except Exception:
        continue

    if len(
        values
    ) == 14:

        if all(
            isinstance(
                x,
                str,
            )
            for x in values
        ):
            candidate_labels.append(
                {
                    "variable_name": name,
                    "labels": values,
                }
            )

print(
    "\nExisting 14-element string label objects:"
)

for item in candidate_labels:

    print(
        f"\n{item['variable_name']}:"
    )

    print(
        item[
            "labels"
        ]
    )

print(
    "\nCell F4.8 complete."
)

print(
    "Next step: use the verified mapped matrix and labels "
    "to construct the revised Figure 4A."
)

In [ ]:
#@title Cell F4.9 - Final combined Figure 4A heatmap
# Purpose:
# Combine the 14 mapped subwindows, 5 supported non-coordinate groups,
# and 2 BH-significant higher-MIC unmapped patterns in one heatmap.
#
# Columns:
#   14 mapped subwindows
#   U02, U03, U05 = supported UNMAPPED groups
#   M05, M07      = supported MULTIMAPPED groups
#   P7319, P2846  = BH-significant unmapped carrier patterns
#
# For the first 19 columns:
#   value = mean Jaccard dissimilarity to 3 matched comparators
#
# For P7319 and P2846:
#   0 = absent
#   1 = present

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# Verify original mapped Figure 4A matrix
# ------------------------------------------------------------

assert high_mean_dissimilarity.shape == (
    16,
    14,
)

assert len(
    region_names
) == 14

assert len(
    high_sample_index
) == 16

assert len(
    high_log2_mic
) == 16

# H01-H16 remain ordered from higher to lower MIC.
assert np.all(
    np.diff(
        high_log2_mic
    )
    <= 0
)

# ------------------------------------------------------------
# Load five supported non-coordinate groups
# ------------------------------------------------------------

NONCOORDINATE_MATRIX_FILE = (
    NB25_DIR
    / "Figure4_noncoordinate_high_MIC_matched_dissimilarity.npz"
)

assert NONCOORDINATE_MATRIX_FILE.exists()

with np.load(
    NONCOORDINATE_MATRIX_FILE
) as archive:

    noncoordinate_matrix = np.asarray(
        archive[
            "noncoordinate_mean_dissimilarity"
        ],
        dtype=float,
    )

    noncoordinate_high_index = np.asarray(
        archive[
            "high_sample_index"
        ],
        dtype=int,
    )

assert noncoordinate_matrix.shape == (
    16,
    5,
)

assert np.array_equal(
    noncoordinate_high_index,
    high_sample_index.astype(
        int
    ),
)

# ------------------------------------------------------------
# Load carrier patterns 7319 and 2846
# ------------------------------------------------------------

PATTERN_WORDS_FILE = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "11_Unitig_Patterns"
    / "11_unique_pattern_words.npz"
)

assert PATTERN_WORDS_FILE.exists()

with np.load(
    PATTERN_WORDS_FILE
) as archive:

    pattern_words = np.asarray(
        archive[
            "unique_words"
        ],
        dtype=np.uint64,
    )

assert pattern_words.shape == (
    504_889,
    3,
)

TARGET_PATTERNS = [
    7319,
    2846,
]


def decode_pattern_words(
    words,
    n_samples=176,
):

    carrier = np.zeros(
        n_samples,
        dtype=np.uint8,
    )

    for sample_index in range(
        n_samples
    ):

        word_index = (
            sample_index
            // 64
        )

        bit_index = (
            sample_index
            % 64
        )

        carrier[
            sample_index
        ] = (
            int(
                words[
                    word_index
                ]
            )
            >> bit_index
        ) & 1

    return carrier


binary_matrix = np.zeros(
    (
        16,
        2,
    ),
    dtype=np.uint8,
)

for pattern_column, pattern_id in enumerate(
    TARGET_PATTERNS
):

    carrier = decode_pattern_words(
        pattern_words[
            pattern_id,
            :
        ]
    )

    binary_matrix[
        :,
        pattern_column,
    ] = carrier[
        high_sample_index.astype(
            int
        )
    ]

# QC from Notebook 17.
assert binary_matrix[
    :,
    0
].sum() == 3

assert binary_matrix[
    :,
    1
].sum() == 2

# H01 and H02 carry both patterns.
assert binary_matrix[
    0,
    :
].tolist() == [
    1,
    1,
]

assert binary_matrix[
    1,
    :
].tolist() == [
    1,
    1,
]

# H15 carries neither.
assert binary_matrix[
    14,
    :
].tolist() == [
    0,
    0,
]

# ------------------------------------------------------------
# Combine all 21 columns
# ------------------------------------------------------------

combined_matrix = np.hstack(
    [
        high_mean_dissimilarity,
        noncoordinate_matrix,
        binary_matrix.astype(
            float
        ),
    ]
)

assert combined_matrix.shape == (
    16,
    21,
)

# Short figure labels.
mapped_names = [
    str(x)
    for x in region_names
]

noncoordinate_names = [
    "U02",
    "U03",
    "U05",
    "M05",
    "M07",
]

pattern_names = [
    "P7319",
    "P2846",
]

combined_names = (
    mapped_names
    + noncoordinate_names
    + pattern_names
)

assert len(
    combined_names
) == 21

print(
    "Figure 4A input QC: PASS"
)

print(
    "Heatmap dimensions:",
    combined_matrix.shape,
)

# ------------------------------------------------------------
# Draw Figure 4A
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(
        11.5,
        6.0,
    )
)

im = ax.imshow(
    combined_matrix,
    cmap="Greys",
    vmin=0,
    vmax=1,
    aspect="auto",
    interpolation="nearest",
)

# Y axis: H01-H16, higher to lower MIC.
ax.set_yticks(
    np.arange(
        16
    )
)

ax.set_yticklabels(
    [
        f"H{i:02d}"
        for i in range(
            1,
            17,
        )
    ]
)

# X axis.
ax.set_xticks(
    np.arange(
        21
    )
)

ax.set_xticklabels(
    combined_names,
    rotation=90,
)

ax.set_ylabel(
    "High-MIC pathogen"
)

# ------------------------------------------------------------
# Visual separators
# ------------------------------------------------------------

# Between mapped parent windows.
for x in [
    3.5,
    6.5,
    10.5,
]:
    ax.axvline(
        x,
        linewidth=0.8,
    )

# Mapped | non-coordinate
ax.axvline(
    13.5,
    linewidth=1.2,
)

# UNMAPPED | MULTIMAPPED
ax.axvline(
    16.5,
    linewidth=0.8,
)

# Non-coordinate | significant patterns
ax.axvline(
    18.5,
    linewidth=1.2,
)

# ------------------------------------------------------------
# Colorbar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.03,
    pad=0.02,
)

cbar.set_label(
    "Mean Jaccard dissimilarity / carrier state"
)

# ------------------------------------------------------------
# Clean journal-style presentation
# ------------------------------------------------------------

ax.tick_params(
    axis="both",
    length=0,
)

for spine in ax.spines.values():
    spine.set_visible(
        False
    )

plt.tight_layout()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

FIGURE_DIR = (
    PROJECT_ROOT
    / "06_Manuscript"
    / "Figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_PNG = (
    FIGURE_DIR
    / "Figure04A_High_MIC_Locus_Distribution_Revised.png"
)

FIGURE_PDF = (
    FIGURE_DIR
    / "Figure04A_High_MIC_Locus_Distribution_Revised.pdf"
)

fig.savefig(
    FIGURE_PNG,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_PDF,
    bbox_inches="tight",
)

plt.show()

print(
    "\nSaved:"
)

print(
    FIGURE_PNG
)

print(
    FIGURE_PDF
)

print(
    "\nCell F4.9 complete."
)

print(
    "Transition: Cell F4.10 will create the revised Figure 4B."
)

In [ ]:
#@title Cell F4.10 - Final Figure 4B mapped and non-coordinate summary
# Purpose:
# Show the quantitative summary for all 19 ablation-supported sequence groups:
#   - 14 mapped subwindows, positioned by MG1655 chromosomal location;
#   - 5 non-coordinate groups, shown separately as categorical groups.
#
# For both group types:
#   Point = median of the 16 high-MIC pathogen mean Jaccard dissimilarities
#           to their 3 matched comparators.
#   Vertical bar = interquartile range across the 16 high-MIC pathogens.
#   Marker size = observed decrease in collective variance fraction after removal.
#
# Marker-size scaling is common across all 19 groups, so mapped and
# non-coordinate ablation effects are directly comparable visually.
#
# P7319 and P2846 are intentionally excluded from Figure 4B because they are
# individual binary carrier states, not ablation-defined sequence groups with
# directly comparable Jaccard-summary and ablation-effect quantities.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Load the five non-coordinate matched-comparator summaries
# generated in Cell F4.6.
# ------------------------------------------------------------

NONCOORDINATE_MATRIX_FILE = (
    NB25_DIR
    / "Figure4_noncoordinate_high_MIC_matched_dissimilarity.npz"
)

NB22_RESULTS = (
    RESULTS_TABLE_DIR
    / "22_carrier_pattern_cluster_matched_random_results.csv"
)

for path in [
    NONCOORDINATE_MATRIX_FILE,
    NB22_RESULTS,
]:
    assert path.exists(), (
        f"Required Figure 4B input not found: {path}"
    )

with np.load(
    NONCOORDINATE_MATRIX_FILE
) as archive:

    noncoordinate_matrix = np.asarray(
        archive[
            "noncoordinate_mean_dissimilarity"
        ],
        dtype=float,
    )

    noncoordinate_high_index = np.asarray(
        archive[
            "high_sample_index"
        ],
        dtype=int,
    )

    noncoordinate_cluster_names = np.asarray(
        archive[
            "cluster_names"
        ]
    ).astype(
        str
    )

assert noncoordinate_matrix.shape == (
    16,
    5,
)

assert np.array_equal(
    noncoordinate_high_index,
    high_sample_index.astype(
        int
    ),
)

expected_noncoordinate_names = np.asarray(
    [
        "UNMAPPED_C02",
        "UNMAPPED_C03",
        "UNMAPPED_C05",
        "MULTIMAPPED_C05",
        "MULTIMAPPED_C07",
    ],
    dtype=str,
)

assert np.array_equal(
    noncoordinate_cluster_names,
    expected_noncoordinate_names,
)

# ------------------------------------------------------------
# Summarise matched-comparator dissimilarity across H01-H16.
# ------------------------------------------------------------

noncoordinate_q1 = np.quantile(
    noncoordinate_matrix,
    0.25,
    axis=0,
)

noncoordinate_median = np.quantile(
    noncoordinate_matrix,
    0.50,
    axis=0,
)

noncoordinate_q3 = np.quantile(
    noncoordinate_matrix,
    0.75,
    axis=0,
)

noncoordinate_lower_error = (
    noncoordinate_median
    - noncoordinate_q1
)

noncoordinate_upper_error = (
    noncoordinate_q3
    - noncoordinate_median
)

assert np.all(
    noncoordinate_lower_error >= -1e-12
)

assert np.all(
    noncoordinate_upper_error >= -1e-12
)

# ------------------------------------------------------------
# Recover the Notebook 22 ablation drops in the same fixed order.
# ------------------------------------------------------------

nb22_results = pd.read_csv(
    NB22_RESULTS
)

for column in [
    "cluster_name",
    "observed_drop_after_removal",
]:
    assert column in nb22_results.columns, (
        f"Missing Notebook 22 column: {column}"
    )

noncoordinate_ablation = (
    nb22_results
    .set_index(
        "cluster_name"
    )
    .loc[
        expected_noncoordinate_names.tolist(),
        "observed_drop_after_removal",
    ]
    .to_numpy(
        dtype=float
    )
)

assert len(
    noncoordinate_ablation
) == 5

# ------------------------------------------------------------
# Use one marker-size scale across all 19 groups.
# ------------------------------------------------------------

mapped_positive_drop = np.maximum(
    mapped_ablation_drop,
    0.0,
)

noncoordinate_positive_drop = np.maximum(
    noncoordinate_ablation,
    0.0,
)

all_positive_drop = np.concatenate(
    [
        mapped_positive_drop,
        noncoordinate_positive_drop,
    ]
)

assert all_positive_drop.max() > 0

marker_size_min = 45.0
marker_size_range = 280.0

mapped_marker_size = (
    marker_size_min
    + marker_size_range
    * (
        mapped_positive_drop
        / all_positive_drop.max()
    )
)

noncoordinate_marker_size = (
    marker_size_min
    + marker_size_range
    * (
        noncoordinate_positive_drop
        / all_positive_drop.max()
    )
)

# ------------------------------------------------------------
# Freeze the exact Figure 4B values for audit and caption writing.
# ------------------------------------------------------------

mapped_plot_table = pd.DataFrame(
    {
        "group_type": "mapped",
        "group_name": final_summary[
            "subwindow_name"
        ].astype(str).to_numpy(),
        "plot_label": final_summary[
            "subwindow_name"
        ].astype(str).to_numpy(),
        "reference_midpoint_Mb": mapped_x,
        "Q1_dissimilarity": mapped_q1,
        "median_dissimilarity": mapped_median,
        "Q3_dissimilarity": mapped_q3,
        "observed_drop_after_removal": mapped_ablation_drop,
    }
)

noncoordinate_short_labels = np.asarray(
    [
        "U02",
        "U03",
        "U05",
        "M05",
        "M07",
    ],
    dtype=str,
)

noncoordinate_plot_table = pd.DataFrame(
    {
        "group_type": "non-coordinate",
        "group_name": expected_noncoordinate_names,
        "plot_label": noncoordinate_short_labels,
        "reference_midpoint_Mb": np.nan,
        "Q1_dissimilarity": noncoordinate_q1,
        "median_dissimilarity": noncoordinate_median,
        "Q3_dissimilarity": noncoordinate_q3,
        "observed_drop_after_removal": noncoordinate_ablation,
    }
)

figure4b_summary = pd.concat(
    [
        mapped_plot_table,
        noncoordinate_plot_table,
    ],
    ignore_index=True,
)

FIG4B_SUMMARY_FILE = (
    RESULTS_TABLE_DIR
    / "Figure4B_mapped_noncoordinate_sequence_summary.csv"
)

figure4b_summary.to_csv(
    FIG4B_SUMMARY_FILE,
    index=False,
)

print(
    "Figure 4B input summary:"
)

display(
    figure4b_summary
)

# ------------------------------------------------------------
# Draw Figure 4B.
# Left: mapped groups on the true MG1655 coordinate scale.
# Right: non-coordinate groups on a separate categorical axis.
# The shared y-axis keeps the dissimilarity summaries directly comparable.
# ------------------------------------------------------------

fig = plt.figure(
    figsize=(10.5, 5.0)
)

grid = fig.add_gridspec(
    1,
    2,
    width_ratios=[
        4.8,
        2.0,
    ],
    wspace=0.08,
)

ax_mapped = fig.add_subplot(
    grid[0, 0]
)

ax_noncoordinate = fig.add_subplot(
    grid[0, 1],
    sharey=ax_mapped,
)

# -------------------------
# Mapped subwindows
# -------------------------

ax_mapped.errorbar(
    mapped_x,
    mapped_median,
    yerr=np.vstack(
        [
            mapped_lower_error,
            mapped_upper_error,
        ]
    ),
    fmt="none",
    ecolor="black",
    capsize=3,
    linewidth=1.0,
    zorder=1,
)

ax_mapped.scatter(
    mapped_x,
    mapped_median,
    s=mapped_marker_size,
    facecolors="0.65",
    edgecolors="black",
    linewidths=0.8,
    zorder=2,
)

MG1655_LENGTH_MB = 4.641652
RIGHT_PADDING_MB = 0.10

ax_mapped.set_xlim(
    0,
    MG1655_LENGTH_MB
    + RIGHT_PADDING_MB,
)

ax_mapped.set_xlabel(
    "MG1655 chromosomal location (Mb)"
)

ax_mapped.set_ylabel(
    "Median Jaccard dissimilarity"
)

# -------------------------
# Non-coordinate groups
# -------------------------

noncoordinate_x = np.arange(
    5,
    dtype=float,
)

ax_noncoordinate.errorbar(
    noncoordinate_x,
    noncoordinate_median,
    yerr=np.vstack(
        [
            noncoordinate_lower_error,
            noncoordinate_upper_error,
        ]
    ),
    fmt="none",
    ecolor="black",
    capsize=3,
    linewidth=1.0,
    zorder=1,
)

ax_noncoordinate.scatter(
    noncoordinate_x,
    noncoordinate_median,
    s=noncoordinate_marker_size,
    facecolors="0.65",
    edgecolors="black",
    linewidths=0.8,
    zorder=2,
)

ax_noncoordinate.set_xlim(
    -0.6,
    4.6,
)

ax_noncoordinate.set_xticks(
    noncoordinate_x
)

ax_noncoordinate.set_xticklabels(
    noncoordinate_short_labels,
    rotation=90,
)

ax_noncoordinate.set_xlabel(
    "Non-coordinate group"
)

# Separate UNMAPPED (U) from MULTIMAPPED (M) without implying coordinates.
ax_noncoordinate.axvline(
    2.5,
    linewidth=0.8,
)

# Use one explicit y-axis range for both panels.
# The upper limit is based on the largest Q3 value across all 19 groups,
# with padding so that the upper error-bar caps are not clipped.
shared_y_max = max(
    float(np.max(mapped_q3)),
    float(np.max(noncoordinate_q3)),
)

shared_y_padding = max(
    0.02,
    0.08 * shared_y_max,
)

ax_mapped.set_ylim(
    0,
    shared_y_max + shared_y_padding,
)

# Shared y-axis: only the mapped panel needs y tick labels.
ax_noncoordinate.tick_params(
    axis="y",
    left=False,
    labelleft=False,
)

# Minimal journal-style presentation.
for ax in [
    ax_mapped,
    ax_noncoordinate,
]:
    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

ax_noncoordinate.spines[
    "left"
].set_visible(
    False
)

fig.subplots_adjust(
    left=0.10,
    right=0.98,
    bottom=0.22,
    top=0.98,
)

# ------------------------------------------------------------
# Save final Figure 4B.
# ------------------------------------------------------------

fig.savefig(
    FIG4B_PNG,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    FIG4B_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print(
    "\nFinal Figure 4B created."
)

print(
    FIG4B_PNG
)

print(
    FIG4B_PDF
)

print(
    "Figure 4B summary table:"
)

print(
    FIG4B_SUMMARY_FILE
)

print(
    "\nFigure 4 complete."
)


In [ ]:
#@title Cell F4.11 - Final combined Figure 4
# Purpose:
# Create the final manuscript Figure 4 directly from the quantitative
# objects generated in Cells F4.9 and F4.10.
#
# Layout:
# - Panel A above Panel B because both panels are wide.
# - Panel B axes are aligned with their corresponding heatmap columns.
# - The mapped chart spans the 14 mapped columns.
# - The non-coordinate chart spans the five non-coordinate columns.
# - No Panel B chart is placed below the two individual-pattern columns.
# - Group separators correspond across Panels A and B.
#
# Important:
# - Uses the binary carrier states already generated and verified in Cell F4.9.
# - No manual changes are made to P7319 or P2846 carrier states.

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Confirm required objects from Cells F4.9 and F4.10
# ------------------------------------------------------------

required_objects = [
    "combined_matrix",
    "combined_names",
    "binary_matrix",
    "mapped_x",
    "mapped_median",
    "mapped_lower_error",
    "mapped_upper_error",
    "mapped_marker_size",
    "noncoordinate_median",
    "noncoordinate_lower_error",
    "noncoordinate_upper_error",
    "noncoordinate_marker_size",
    "noncoordinate_short_labels",
    "shared_y_max",
    "shared_y_padding",
    "FIGURE_DIR",
]

for object_name in required_objects:
    assert object_name in globals(), (
        f"Required object not found: {object_name}. "
        "Run Cells F4.9 and F4.10 first."
    )

# ------------------------------------------------------------
# Verify carrier states and matrix dimensions
# ------------------------------------------------------------

assert binary_matrix.shape == (16, 2)

# P7319 has three high-MIC carriers.
assert binary_matrix[:, 0].sum() == 3

# P2846 has two high-MIC carriers.
assert binary_matrix[:, 1].sum() == 2

# H01 and H02 carry both patterns.
assert binary_matrix[0, :].tolist() == [1, 1]
assert binary_matrix[1, :].tolist() == [1, 1]

# H13 carries P7319 but not P2846.
assert binary_matrix[12, :].tolist() == [1, 0]

# H15 carries neither pattern.
assert binary_matrix[14, :].tolist() == [0, 0]

assert combined_matrix.shape == (16, 21)
assert len(combined_names) == 21
assert len(mapped_x) == 14

# ------------------------------------------------------------
# Heatmap column structure
# ------------------------------------------------------------

N_MAPPED_COLUMNS = 14
N_NONCOORDINATE_COLUMNS = 5
N_PATTERN_COLUMNS = 2
N_TOTAL_COLUMNS = 21

assert (
    N_MAPPED_COLUMNS
    + N_NONCOORDINATE_COLUMNS
    + N_PATTERN_COLUMNS
    == N_TOTAL_COLUMNS
)

# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

OUTPUT_PNG = FIGURE_DIR / "Fig4.png"
OUTPUT_PDF = FIGURE_DIR / "Fig4.pdf"

# ------------------------------------------------------------
# Main figure
# ------------------------------------------------------------

fig = plt.figure(
    figsize=(12.0, 10.5)
)

outer_grid = fig.add_gridspec(
    2,
    1,
    height_ratios=[1.15, 1.0],
    hspace=0.42,
)

# ============================================================
# Panel A
# ============================================================

ax_a = fig.add_subplot(
    outer_grid[0, 0]
)

im = ax_a.imshow(
    combined_matrix,
    cmap="Greys",
    vmin=0,
    vmax=1,
    aspect="auto",
    interpolation="nearest",
)

ax_a.set_yticks(
    np.arange(16)
)

ax_a.set_yticklabels(
    [
        f"H{i:02d}"
        for i in range(1, 17)
    ]
)

ax_a.set_xticks(
    np.arange(21)
)

ax_a.set_xticklabels(
    combined_names,
    rotation=90,
)

ax_a.set_ylabel(
    "High-MIC pathogen"
)

ax_a.set_title(
    "A. Matched-comparator sequence differences",
    fontsize=13,
    pad=12,
)

# ------------------------------------------------------------
# Panel A column-group separators
# ------------------------------------------------------------

# Between mapped parent windows
for x in [3.5, 6.5, 10.5]:
    ax_a.axvline(
        x,
        linewidth=0.8,
        color="black",
    )

# Mapped | non-coordinate
ax_a.axvline(
    13.5,
    linewidth=1.2,
    color="black",
)

# UNMAPPED | MULTIMAPPED
ax_a.axvline(
    16.5,
    linewidth=0.8,
    color="black",
)

# Non-coordinate | individual patterns
ax_a.axvline(
    18.5,
    linewidth=1.2,
    color="black",
)

# ------------------------------------------------------------
# Colorbar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=ax_a,
    fraction=0.025,
    pad=0.02,
)

cbar.set_label(
    "Mean Jaccard dissimilarity / carrier state"
)

ax_a.tick_params(
    axis="both",
    length=0,
)

for spine in ax_a.spines.values():
    spine.set_visible(False)

# ------------------------------------------------------------
# Establish final margins before positioning Panel B
# ------------------------------------------------------------

fig.subplots_adjust(
    left=0.09,
    right=0.96,
    bottom=0.08,
    top=0.96,
)

fig.canvas.draw()

heatmap_position = ax_a.get_position()
bottom_position = outer_grid[1, 0].get_position(fig)

# ------------------------------------------------------------
# Calculate Panel B positions from the heatmap columns
# ------------------------------------------------------------

heatmap_column_width = (
    heatmap_position.width
    / N_TOTAL_COLUMNS
)

mapped_left = heatmap_position.x0

mapped_right = (
    heatmap_position.x0
    + N_MAPPED_COLUMNS * heatmap_column_width
)

noncoordinate_left = mapped_right

noncoordinate_right = (
    heatmap_position.x0
    + (
        N_MAPPED_COLUMNS
        + N_NONCOORDINATE_COLUMNS
    )
    * heatmap_column_width
)

# Small visual gap centred on the mapped/non-coordinate boundary.
PANEL_B_GAP = 0.012

mapped_axis_position = [
    mapped_left,
    bottom_position.y0,
    mapped_right
    - mapped_left
    - PANEL_B_GAP / 2,
    bottom_position.height,
]

noncoordinate_axis_position = [
    noncoordinate_left
    + PANEL_B_GAP / 2,
    bottom_position.y0,
    noncoordinate_right
    - noncoordinate_left
    - PANEL_B_GAP / 2,
    bottom_position.height,
]

# ============================================================
# Panel B
# ============================================================

ax_b_mapped = fig.add_axes(
    mapped_axis_position
)

ax_b_noncoordinate = fig.add_axes(
    noncoordinate_axis_position,
    sharey=ax_b_mapped,
)

# ------------------------------------------------------------
# Separator aligned with mapped | non-coordinate boundary
# ------------------------------------------------------------

panel_b_group_separator = plt.Line2D(
    [mapped_right, mapped_right],
    [
        bottom_position.y0,
        bottom_position.y1,
    ],
    transform=fig.transFigure,
    color="black",
    linewidth=1.2,
    zorder=10,
)

fig.add_artist(
    panel_b_group_separator
)

# ------------------------------------------------------------
# Panel B title
# ------------------------------------------------------------

panel_b_centre = (
    mapped_left
    + noncoordinate_right
) / 2

fig.text(
    panel_b_centre,
    bottom_position.y1 + 0.022,
    "B. Mapped and non-coordinate sequence-group summary",
    ha="center",
    va="bottom",
    fontsize=13,
)

# ------------------------------------------------------------
# Mapped subwindows
# ------------------------------------------------------------

ax_b_mapped.errorbar(
    mapped_x,
    mapped_median,
    yerr=np.vstack(
        [
            mapped_lower_error,
            mapped_upper_error,
        ]
    ),
    fmt="none",
    ecolor="black",
    capsize=3,
    linewidth=1.0,
    zorder=1,
)

ax_b_mapped.scatter(
    mapped_x,
    mapped_median,
    s=mapped_marker_size,
    facecolors="0.65",
    edgecolors="black",
    linewidths=0.8,
    zorder=2,
)

# ------------------------------------------------------------
# Faint separators between mapped parent-window groups
# ------------------------------------------------------------

# W01 contains positions 0-3.
# W04 contains positions 4-6.
# W07 contains positions 7-10.
# W10 contains positions 11-13.
mapped_group_end_indices = [3, 6, 10]

mapped_group_separator_positions = [
    (
        mapped_x[end_index]
        + mapped_x[end_index + 1]
    ) / 2
    for end_index in mapped_group_end_indices
]

for separator_x in mapped_group_separator_positions:
    ax_b_mapped.axvline(
        separator_x,
        linewidth=0.8,
        color="0.75",
        zorder=0,
    )

MG1655_LENGTH_MB = 4.641652
RIGHT_PADDING_MB = 0.10

ax_b_mapped.set_xlim(
    0,
    MG1655_LENGTH_MB + RIGHT_PADDING_MB,
)

ax_b_mapped.set_xlabel(
    "MG1655 chromosomal location (Mb)"
)

ax_b_mapped.set_ylabel(
    "Median Jaccard dissimilarity"
)

# ------------------------------------------------------------
# Non-coordinate groups
# ------------------------------------------------------------

noncoordinate_x = np.arange(
    5,
    dtype=float,
)

ax_b_noncoordinate.errorbar(
    noncoordinate_x,
    noncoordinate_median,
    yerr=np.vstack(
        [
            noncoordinate_lower_error,
            noncoordinate_upper_error,
        ]
    ),
    fmt="none",
    ecolor="black",
    capsize=3,
    linewidth=1.0,
    zorder=1,
)

ax_b_noncoordinate.scatter(
    noncoordinate_x,
    noncoordinate_median,
    s=noncoordinate_marker_size,
    facecolors="0.65",
    edgecolors="black",
    linewidths=0.8,
    zorder=2,
)

ax_b_noncoordinate.set_xlim(
    -0.6,
    4.6,
)

ax_b_noncoordinate.set_xticks(
    noncoordinate_x
)

ax_b_noncoordinate.set_xticklabels(
    noncoordinate_short_labels,
    rotation=90,
)

ax_b_noncoordinate.set_xlabel(
    "Non-coordinate group"
)

# Separate U groups from M groups.
ax_b_noncoordinate.axvline(
    2.5,
    linewidth=0.8,
    color="black",
)

# ------------------------------------------------------------
# Shared y-axis
# ------------------------------------------------------------

ax_b_mapped.set_ylim(
    0,
    shared_y_max + shared_y_padding,
)

ax_b_noncoordinate.tick_params(
    axis="y",
    left=False,
    labelleft=False,
)

# ------------------------------------------------------------
# Clean journal-style presentation
# ------------------------------------------------------------

for ax in [
    ax_b_mapped,
    ax_b_noncoordinate,
]:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

ax_b_noncoordinate.spines["left"].set_visible(False)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

fig.savefig(
    OUTPUT_PNG,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(
    "Final Figure 4 created."
)

print(
    "Saved PNG:",
    OUTPUT_PNG,
)

print(
    "Saved PDF:",
    OUTPUT_PDF,
)